## 📋 Post-Generation Checklist

✅ **Before training on dataset_expert_v4.json:**

1. **Inspect the output file**
   - Open `dataset_expert_v4.json` in your editor
   - Search for `<think>` to verify tags are present
   - Spot-check 5-10 records manually to ensure quality

2. **Check the summary metrics**
   - Run the summary cell above and verify:
     - `<think>` tag validation success rate > 90%
     - Total records V4 > V3 (should be 50-60+ if all went well)
     - No concept has 0 records

3. **If validation failed**
   - Check your OpenAI API key is valid
   - Verify quota/credits in your OpenAI account
   - Re-run generation with `MODEL = "gpt-3.5-turbo"` (cheaper, still good)
   - Or manually inspect rejected records in the output above

---

## 🚀 Next Steps: Two Paths Forward

### Path A: Quick Test (Recommended first)
Use your current V3-trained model with LM Studio:
```bash
ollama serve mpidia-lora_sauvegarde
# Or launch LM Studio with current adapter
```
Test with a few queries to see what Apex V3 can already do.

### Path B: Ultimate Training (After Path A)
Once V4 is validated, retrain on Colab:
1. Open `colab_finetune.ipynb`
2. Change dataset path: `dataset_apex_v1.json` → `dataset_expert_v4.json`
3. Run full fine-tuning pipeline
4. Download the new adapter weights
5. Deploy as your "SRE Super Engineer" Apex V4

In [5]:
def build_strict_variation_prompt(concept_key: str, sample_instruction: str, sample_output: str) -> str:
    """
    Build a prompt that STRICTLY requires <think>...</think> tags in output.
    This is critical for Apex's reasoning ability.
    """
    return f"""You are a JSON dataset augmentation expert. Generate exactly 10 diverse variations.

**Original Example (Concept: {concept_key}):**
Instruction: {sample_instruction}
Output: {sample_output}

**CRITICAL REQUIREMENTS FOR EACH VARIATION:**
1. Change numbers/values completely - NOT just +-10 percent variations
2. Vary phrasing - use different tones (formal, casual, urgent, technical, simplified)
3. Different real-world context - different companies, time scales, scenarios
4. Preserve core logic and reasoning pattern
5. MUST include <think>...</think> tags in EVERY output - this is mandatory, not optional

**EXACT OUTPUT FORMAT (Valid JSON array with 10 objects):**
[
  {{
    "instruction": "<variation 1 - different instruction>",
    "output": "<think>\\n<reasoning process here>\\n</think>\\n<final answer here>"
  }},
  {{
    "instruction": "<variation 2>",
    "output": "<think>\\n<reasoning>\\n</think>\\n<answer>"
  }},
  ...
]

**DO NOT return markdown code blocks, DO NOT include explanations, ONLY return the JSON array.**
"""

print("Updated prompt template with STRICT <think> tag requirements")
print("Prompt target: 10 variations per concept")

Updated prompt template with STRICT <think> tag requirements
Prompt target: 10 variations per concept


In [9]:
# STRICT VALIDATION: Re-validate all generated records for <think> tags
print("🔍 STRICT VALIDATION: Checking <think> tags in all generated records\n")

think_tag_failures = []
validated_with_think = []

if not validated_records:
    print("⚠️ No generated records to validate (validated_records is empty).")
    print("This usually means generation failed upstream (for example API quota or auth issues).")
else:
    for i, record in enumerate(validated_records):
        instruction = record.get("instruction", "").strip()
        output = record.get("output", "").strip()

        is_valid, error = validate_think_tags(output)

        if not is_valid:
            think_tag_failures.append({
                "index": i,
                "instruction": instruction[:60],
                "error": error,
            })
            print(f"❌ Record {i}: {error}")
            print(f"   Instruction: {instruction[:70]}...")
            print(f"   Output snippet: {output[:100]}...\n")
        else:
            validated_with_think.append(record)
            print(f"✅ Record {i}: Valid <think> tags present")

print(f"\n📊 <think> Validation Report:")
print(f"  ✅ Valid records: {len(validated_with_think)}")
print(f"  ❌ Failed records: {len(think_tag_failures)}")

if validated_records:
    success_rate = 100 * len(validated_with_think) / len(validated_records)
    print(f"  Success rate: {success_rate:.1f}%")
else:
    print("  Success rate: N/A (no records generated)")

if think_tag_failures:
    print(f"\n⚠️ WARNING: {len(think_tag_failures)} records are missing <think> tags!")
    print("These will NOT be used in the final dataset.")
    print("Consider re-running generation with a stricter prompt.\n")

# Use only records with valid <think> tags
validated_records = validated_with_think

🔍 STRICT VALIDATION: Checking <think> tags in all generated records

✅ Record 0: Valid <think> tags present
✅ Record 1: Valid <think> tags present
✅ Record 2: Valid <think> tags present
✅ Record 3: Valid <think> tags present
✅ Record 4: Valid <think> tags present
✅ Record 5: Valid <think> tags present
✅ Record 6: Valid <think> tags present
✅ Record 7: Valid <think> tags present
✅ Record 8: Valid <think> tags present
✅ Record 9: Valid <think> tags present
✅ Record 10: Valid <think> tags present
✅ Record 11: Valid <think> tags present
✅ Record 12: Valid <think> tags present
✅ Record 13: Valid <think> tags present
✅ Record 14: Valid <think> tags present
✅ Record 15: Valid <think> tags present
✅ Record 16: Valid <think> tags present
✅ Record 17: Valid <think> tags present
✅ Record 18: Valid <think> tags present
✅ Record 19: Valid <think> tags present
✅ Record 20: Valid <think> tags present
✅ Record 21: Valid <think> tags present
✅ Record 22: Valid <think> tags present
✅ Record 23: Valid <t

In [8]:
def validate_think_tags(output: str) -> tuple[bool, str]:
    """
    Strictly validate that output contains proper <think>...</think> tags.
    Returns (is_valid, error_message)
    """
    output = output.strip()
    
    # Check for opening and closing tags
    has_open = "<think>" in output
    has_close = "</think>" in output
    
    if not has_open or not has_close:
        missing = []
        if not has_open:
            missing.append("<think>")
        if not has_close:
            missing.append("</think>")
        return False, f"Missing tags: {', '.join(missing)}"
    
    # Check that opening comes before closing
    open_idx = output.find("<think>")
    close_idx = output.find("</think>")
    if open_idx >= close_idx:
        return False, "</think> comes before <think> or tags overlap"
    
    # Check for content between tags
    think_content = output[open_idx + 7:close_idx].strip()
    if len(think_content) < 5:
        return False, "<think> content too short (< 5 chars)"
    
    return True, ""

# Test the validator
test_outputs = [
    "<think>\nGood output\n</think>\nFinal answer",
    "Bad: no think tags",
    "<think>\n</think>",  # Empty
    "Multiple <think> tags are <think> bad",
]

print("🧪 Testing <think> validator:\n")
for i, test in enumerate(test_outputs, 1):
    is_valid, error = validate_think_tags(test)
    status = "✅" if is_valid else "❌"
    print(f"{status} Test {i}: {error if error else 'Valid'}")
    print(f"   Input: {test[:60]}...\n")

🧪 Testing <think> validator:

✅ Test 1: Valid
   Input: <think>
Good output
</think>
Final answer...

❌ Test 2: Missing tags: <think>, </think>
   Input: Bad: no think tags...

❌ Test 3: <think> content too short (< 5 chars)
   Input: <think>
</think>...

❌ Test 4: Missing tags: </think>
   Input: Multiple <think> tags are <think> bad...



## ⚠️ CRITICAL: Validate <think> Tags

This cell ensures all generated outputs include properly formatted `<think>...</think>` tags.
This is where Apex's reasoning power comes from—if tags are missing, the model loses its "thinking" capability.

In [11]:
# Save to dataset_expert_v4.json
output_path = Path("dataset_expert_v4.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(final_dataset)} records to {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.1f} KB")

# Preview sample records
print(f"\n📋 Sample Records from V4 Dataset:\n")

# Show one original
print("Original (V3):")
original_sample = [r for r in final_dataset if "hard-fix #1" in r.get("instruction", "")]
if original_sample:
    print(f"  Instruction: {original_sample[0]['instruction'][:80]}...")
    print(f"  Output: {original_sample[0]['output'][:80]}...")

# Show one synthetic (if available)
print("\nSynthetic (Generated):")
synthetic_sample = [r for r in final_dataset if "hard-fix #" not in r.get("instruction", "")]
if synthetic_sample:
    print(f"  Instruction: {synthetic_sample[0]['instruction'][:80]}...")
    print(f"  Output: {synthetic_sample[0]['output'][:80]}...")

print(f"\n✨ Dataset V4 is ready for training!")
print(f"   V3 size: {len(raw_data)} (heavily duplicated)")
print(f"   V4 size: {len(final_dataset)} (diverse, synthetic-augmented)")

✅ Saved 255 records to dataset_expert_v4.json
   File size: 117.5 KB

📋 Sample Records from V4 Dataset:

Original (V3):
  Instruction: Calculus hard-fix #1: For f(x)=x^3-6x^2+9x+1, find critical points and classify ...
  Output: <think>
Compute f'(x)=3x^2-12x+9 and solve f'(x)=0.
Use f''(x)=6x-12 to classify...

Synthetic (Generated):
  Instruction: Urgent: Determine turning points for g(y)=y^3-15y^2+48y-20 and categorize them....
  Output: <think> Evaluate g'(y)=3y^2-30y+48 and solve g'(y)=0. Using g''(y)=6y-30, classi...

✨ Dataset V4 is ready for training!
   V3 size: 180 (heavily duplicated)
   V4 size: 255 (diverse, synthetic-augmented)


In [10]:
# Deduplicate: check against original dataset and generated records
seen_instructions = set()
final_dataset = []


def _instruction_fingerprint(text: str) -> str:
    """Normalize full instruction text to avoid false duplicates."""
    return " ".join((text or "").strip().lower().split())


# Add original records first
for record in raw_data:
    instruction_key = _instruction_fingerprint(record.get("instruction", ""))
    seen_instructions.add(instruction_key)
    final_dataset.append(record)

print(f"Original dataset size: {len(final_dataset)}")

# Add validated synthetic records, skipping only true full-text duplicates
duplicates_skipped = 0
for record in validated_records:
    instruction_key = _instruction_fingerprint(record.get("instruction", ""))

    if instruction_key and instruction_key not in seen_instructions:
        seen_instructions.add(instruction_key)
        final_dataset.append(record)
    else:
        duplicates_skipped += 1

print(f"Synthetic records added: {len(validated_records) - duplicates_skipped}")
print(f"Duplicates skipped: {duplicates_skipped}")
print(f"Final dataset size: {len(final_dataset)}")

# Summary by concept
concept_counts = defaultdict(int)
for record in final_dataset:
    instr = record["instruction"]
    match = re.match(r'([^:]+):', instr)
    concept = match.group(1).strip() if match else "Unknown"
    concept_counts[concept] += 1

print("\nFinal dataset breakdown by concept:")
for concept in sorted(concept_counts.keys()):
    count = concept_counts[concept]
    print(f"  - {concept}: {count} records")

Original dataset size: 180
Synthetic records added: 75
Duplicates skipped: 0
Final dataset size: 255

Final dataset breakdown by concept:
  - ABC Logistics: 1 records
  - Analyzing quarterly results: 1 records
  - Calculus hard-fix #1: 1 records
  - Calculus hard-fix #10: 1 records
  - Calculus hard-fix #11: 1 records
  - Calculus hard-fix #12: 1 records
  - Calculus hard-fix #13: 1 records
  - Calculus hard-fix #14: 1 records
  - Calculus hard-fix #15: 1 records
  - Calculus hard-fix #16: 1 records
  - Calculus hard-fix #17: 1 records
  - Calculus hard-fix #18: 1 records
  - Calculus hard-fix #19: 1 records
  - Calculus hard-fix #2: 1 records
  - Calculus hard-fix #20: 1 records
  - Calculus hard-fix #3: 1 records
  - Calculus hard-fix #4: 1 records
  - Calculus hard-fix #5: 1 records
  - Calculus hard-fix #6: 1 records
  - Calculus hard-fix #7: 1 records
  - Calculus hard-fix #8: 1 records
  - Calculus hard-fix #9: 1 records
  - Casual: 1 records
  - Code Repair Task #12: 1 records
 

In [7]:
# Parse, normalize, and validate records
validated_records = []
rejected_records = []

for record in generated_records:
    instruction = (record.get("instruction") or "").strip()
    output = (record.get("output") or "").strip()
    
    # Validation checks
    errors = []
    if not instruction:
        errors.append("empty instruction")
    if not output:
        errors.append("empty output")
    if len(instruction) < 10:
        errors.append("instruction too short")
    if len(output) < 5:
        errors.append("output too short")
    
    if errors:
        rejected_records.append({
            "record": record,
            "errors": errors
        })
        continue
    
    # Normalize whitespace
    instruction = " ".join(instruction.split())
    output = " ".join(output.split())
    
    # Ensure output has <think> tags if not present
    if "<think>" not in output:
        # Simple output - wrap in think tags
        output = f"<think>\n{output}\n</think>" if "\n" in output or len(output) > 200 else output
    
    validated_records.append({
        "instruction": instruction,
        "output": output
    })

print(f"✅ Validation Results:")
print(f"  Valid records: {len(validated_records)}")
print(f"  Rejected records: {len(rejected_records)}")
if rejected_records:
    print(f"\n  Sample rejections:")
    for item in rejected_records[:3]:
        print(f"    - {item['record'].get('source_concept', 'unknown')}: {'; '.join(item['errors'])}")

✅ Validation Results:
  Valid records: 75
  Rejected records: 0


In [6]:
import time
import re as _re


# -- JSON repair helpers -------------------------------------------------------

def _extract_json_array(text: str) -> str:
    """Best-effort extraction of a JSON array from model text output."""
    raw = (text or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.replace("json", "", 1).strip()

    # Direct array
    start = raw.find("[")
    end = raw.rfind("]")
    if start != -1 and end != -1 and end > start:
        return raw[start : end + 1]

    # Sometimes model wraps with {"variations": [...]}
    key = '"variations"'
    kidx = raw.find(key)
    if kidx != -1:
        arr_start = raw.find("[", kidx)
        arr_end = raw.rfind("]")
        if arr_start != -1 and arr_end != -1 and arr_end > arr_start:
            return raw[arr_start : arr_end + 1]

    return raw


def _repair_json(text: str) -> str:
    """
    Light-touch JSON repair for common LLM mistakes:
      - Trailing commas before ] or }
      - Unescaped newlines inside string values
      - Truncated arrays (no closing ])
    """
    text = _re.sub(r",\s*([\]}])", r"\1", text)

    def _fix_newlines(m):
        return m.group(0).replace("\n", "\\n").replace("\r", "")

    text = _re.sub(r'"(?:[^"\\]|\\.)*"', _fix_newlines, text, flags=_re.DOTALL)

    if text.count("[") > text.count("]"):
        text = text.rstrip().rstrip(",") + "\n]"

    return text


def _salvage_variations(response_text: str):
    """Recover instruction/output pairs from broken JSON-like text."""
    text = (response_text or "").replace("\r", "")
    pattern = _re.compile(
        r'"instruction"\s*:\s*"(?P<instruction>(?:[^"\\]|\\.)*)"\s*,\s*"output"\s*:\s*"(?P<output>(?:[^"\\]|\\.)*)"',
        flags=_re.DOTALL,
    )

    recovered = []
    for m in pattern.finditer(text):
        instr_escaped = m.group("instruction")
        out_escaped = m.group("output")
        try:
            instruction = json.loads(f'"{instr_escaped}"')
            output = json.loads(f'"{out_escaped}"')
        except Exception:
            instruction = instr_escaped.encode("utf-8", "ignore").decode("unicode_escape", "ignore")
            output = out_escaped.encode("utf-8", "ignore").decode("unicode_escape", "ignore")

        recovered.append({"instruction": instruction, "output": output})

    return recovered


def _parse_variations(response_text: str):
    """Parse text into a list of variations with multiple fallbacks."""
    json_text = _extract_json_array(response_text)
    try:
        result = json.loads(json_text)
    except json.JSONDecodeError:
        repaired = _repair_json(json_text)
        try:
            result = json.loads(repaired)
        except json.JSONDecodeError:
            result = _salvage_variations(response_text)

    if not isinstance(result, list):
        raise ValueError("Parsed value is not a JSON array")

    # Keep only dict-like items
    result = [item for item in result if isinstance(item, dict)]
    return result


def _coerce_to_json_with_lmstudio(raw_response: str) -> str:
    """Ask LM Studio to convert raw text into strict JSON array."""
    repair_prompt = f"""Convert the following text into strict JSON only.

Rules:
- Return ONLY a JSON array.
- Each item is an object with keys: instruction, output.
- Preserve <think>...</think> tags in output.
- No markdown. No explanations.

Text to convert:
{raw_response}
"""
    return _call_lmstudio(repair_prompt, temperature=0.2)


# -- API callers ---------------------------------------------------------------

def _call_openai(prompt: str) -> str:
    assert client is not None
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


def _call_lmstudio(prompt: str, temperature: float = 0.7, max_tokens: int | None = None) -> str:
    response = lmstudio_client.chat.completions.create(
        model=LMSTUDIO_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens or MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


def _single_variation_prompt(concept_key: str, sample_instruction: str, sample_output: str, index: int) -> str:
    return f"""Generate exactly ONE variation for this concept.

Concept: {concept_key}
Original instruction: {sample_instruction}
Original output: {sample_output}

Requirements:
- Keep concept logic but change numbers/context/phrasing.
- Output must include <think>...</think>.
- Return STRICT JSON object only:
{{"instruction":"...","output":"<think>...\\n</think>..."}}
- No markdown, no explanations.

Variation number: {index}
"""


def _generate_variations_single_fallback(concept_key: str, sample_instruction: str, sample_output: str, target_count: int):
    """Reliable fallback: ask LM Studio for one JSON object at a time."""
    out = []
    seen = set()
    for idx in range(1, target_count + 1):
        prompt = _single_variation_prompt(concept_key, sample_instruction, sample_output, idx)
        try:
            text = _call_lmstudio(prompt, temperature=0.6, max_tokens=1200)
            # parse as single object first
            obj = None
            try:
                obj = json.loads((text or "").strip())
                if not isinstance(obj, dict):
                    obj = None
            except Exception:
                pass

            # if not a JSON object, try list parser salvage
            if obj is None:
                parsed = _parse_variations(text)
                if parsed:
                    obj = parsed[0]

            if not obj:
                continue

            instr = (obj.get("instruction") or "").strip()
            resp = (obj.get("output") or "").strip()
            key = " ".join(instr.lower().split())
            if instr and resp and key and key not in seen:
                seen.add(key)
                out.append({"instruction": instr, "output": resp})
        except Exception:
            continue

        time.sleep(0.2)

    return out


# -- Generation with retry -----------------------------------------------------

MAX_RETRIES = 3
TARGET_VARIATIONS_PER_CONCEPT = 10

generated_records = []
failed_concepts = []

for concept_key, samples in sorted(concepts_dict.items()):
    original_instruction = samples[0]["original_instruction"]
    original_output = samples[0]["output"]

    print(f"\nGenerating variations for: {concept_key}")
    prompt = build_strict_variation_prompt(concept_key, original_instruction, original_output)

    variations = None
    provider_used = "none"

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if USE_OPENAI:
                provider_used = "openai"
                response_text = _call_openai(prompt)
            else:
                raise RuntimeError("OpenAI disabled.")
        except Exception as openai_err:
            print(f"  OpenAI attempt {attempt}: {str(openai_err)[:120]}")
            try:
                provider_used = "lmstudio"
                response_text = _call_lmstudio(prompt, temperature=0.7)
                print("  Fallback to LM Studio successful")
            except Exception as lmstudio_err:
                print(f"  LM Studio attempt {attempt} failed: {str(lmstudio_err)[:120]}")
                if attempt < MAX_RETRIES:
                    time.sleep(2)
                continue

        try:
            variations = _parse_variations(response_text)
            if len(variations) < 1:
                raise ValueError("Empty variation list")
            if len(variations) != TARGET_VARIATIONS_PER_CONCEPT:
                print(f"  Warning: got {len(variations)} variations (target {TARGET_VARIATIONS_PER_CONCEPT})")
            print(f"  Added {len(variations)} variations via {provider_used} (attempt {attempt})")
            break
        except Exception as parse_err:
            print(f"  Parse attempt {attempt}: {str(parse_err)[:120]}")
            preview = (response_text or "").strip()[:220].replace("\n", " ")
            print(f"  Raw response preview: {preview}")
            variations = None

            if provider_used == "lmstudio":
                try:
                    repaired_text = _coerce_to_json_with_lmstudio(response_text)
                    repaired_variations = _parse_variations(repaired_text)
                    if repaired_variations:
                        variations = repaired_variations
                        print(f"  Recovery pass succeeded with {len(variations)} variations")
                        break
                    print("  Recovery pass returned an empty list")
                except Exception as repair_err:
                    print(f"  Recovery pass failed: {str(repair_err)[:120]}")

            if attempt < MAX_RETRIES:
                time.sleep(2)

    # Final safety fallback: one variation at a time (most reliable with local models)
    if not variations:
        print("  Switching to single-variation fallback mode")
        variations = _generate_variations_single_fallback(
            concept_key,
            original_instruction,
            original_output,
            TARGET_VARIATIONS_PER_CONCEPT,
        )
        if variations:
            provider_used = "lmstudio-single"
            print(f"  Single fallback recovered {len(variations)} variations")

    if not variations:
        print(f"  All attempts failed for: {concept_key}")
        failed_concepts.append(concept_key)
        continue

    for i, var in enumerate(variations):
        generated_records.append(
            {
                "instruction": (var.get("instruction") or "").strip(),
                "output": (var.get("output") or "").strip(),
                "source_concept": concept_key,
                "variation_idx": i + 1,
                "provider": provider_used,
            }
        )

    time.sleep(0.5)

print("\nGeneration Summary:")
print(f"  Total variations generated : {len(generated_records)}")
print(f"  Failed concepts            : {len(failed_concepts)}")
if failed_concepts:
    print(f"  Failed list: {failed_concepts}")


Generating variations for: Calculus hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: Code hard-fix
  Added 5 variations via openai (attempt 1)

Generating variations for: Instruction hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: Metrics hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: Queue hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: SRE hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: Safety hard-fix
  Added 10 variations via openai (attempt 1)

Generating variations for: Translation hard-fix
  Added 10 variations via openai (attempt 1)

Generation Summary:
  Total variations generated : 75
  Failed concepts            : 0


In [2]:
# Configure OpenAI + LM Studio clients (hybrid mode)
import os
from openai import OpenAI
from pathlib import Path


def _read_env_value(key: str, env_path: str = ".env") -> str | None:
    """Read KEY=value from .env without extra dependencies."""
    p = Path(env_path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        k, v = stripped.split("=", 1)
        if k.strip() == key:
            return v.strip().strip('"').strip("'")
    return None


# OpenAI (primary)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or _read_env_value("OPENAI_API_KEY")
USE_OPENAI = bool(OPENAI_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY) if USE_OPENAI else None
OPENAI_MODEL = os.getenv("APEX_OPENAI_MODEL") or _read_env_value("APEX_OPENAI_MODEL") or "gpt-4o"

# LM Studio (local fallback, OpenAI-compatible API)
LMSTUDIO_BASE_URL = (
    os.getenv("APEX_LMSTUDIO_BASE_URL")
    or _read_env_value("APEX_LMSTUDIO_BASE_URL")
    or "http://localhost:1234/v1"
).rstrip("/")
LMSTUDIO_API_KEY = os.getenv("APEX_LMSTUDIO_API_KEY") or _read_env_value("APEX_LMSTUDIO_API_KEY") or "lm-studio"
LMSTUDIO_MODEL = os.getenv("APEX_LMSTUDIO_MODEL") or _read_env_value("APEX_LMSTUDIO_MODEL") or "phi-4"
lmstudio_client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LMSTUDIO_API_KEY)

# Generation settings
TEMPERATURE = 0.9
MAX_TOKENS = 2000

print("📡 Generation backend configuration:")
print(f"  OpenAI key present: {USE_OPENAI}")
print(f"  OpenAI key source: {'env/.env' if USE_OPENAI else 'missing'}")
if USE_OPENAI:
    print(f"  OpenAI model (primary): {OPENAI_MODEL}")
else:
    print("  OpenAI model (primary): disabled (missing key)")
print(f"  LM Studio URL (fallback): {LMSTUDIO_BASE_URL}")
print(f"  LM Studio model (fallback): {LMSTUDIO_MODEL}")
print(f"  LM Studio api_key mode: {'custom' if LMSTUDIO_API_KEY != 'lm-studio' else 'dummy default'}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Max tokens: {MAX_TOKENS}")

📡 Generation backend configuration:
  OpenAI key present: True
  OpenAI key source: env/.env
  OpenAI model (primary): gpt-4o
  LM Studio URL (fallback): http://localhost:1234/v1
  LM Studio model (fallback): phi-4
  LM Studio api_key mode: dummy default
  Temperature: 0.9
  Max tokens: 2000


In [ ]:
def build_variation_prompt(concept_key: str, sample_instruction: str, sample_output: str) -> str:\n    \"\"\"\n    Build a prompt that asks GPT to generate 5 diverse variations.\n    Force diversity by explicitly requesting:\n    - Different numbers/values\n    - Different phrasing and tone\n    - Different real-world contexts\n    \"\"\"\n    return f\"\"\"You are a dataset augmentation expert. Your task is to generate 5 highly diverse variations of this training example, avoiding overfitting.\n\n**Original Example (Concept: {concept_key}):**\nInstruction: {sample_instruction}\nOutput: {sample_output}\n\n**Requirements for each variation:**\n1. **Change numbers/values completely** - if original has 120 req/s, use 80, 150, 200, etc. Not just ±10%\n2. **Vary phrasing and tone** - sometimes formal, sometimes casual, sometimes urgent; change question structure\n3. **Create real-world context** - reference different companies, different scenarios, different time scales\n4. **Preserve core logic** - the reasoning pattern must remain the same\n5. **Maintain JSON structure** - output MUST be valid JSON\n\n**Output format:** Return a JSON array with exactly 5 objects, each with \"instruction\" and \"output\" keys.\n\nExample output format:\n[\n  {{\n    \"instruction\": \"<variation 1 instruction>\",\n    \"output\": \"<variation 1 output with <think> tags>\"\n  }},\n  {{\n    \"instruction\": \"<variation 2 instruction>\",\n    \"output\": \"<variation 2 output with <think> tags>\"\n  }},\n  ...\n]\n\nGenerate now:\"\"\"\n\nprint(\"✅ Variation prompt template built\")\nprint(f\"\\nExample prompt preview (first 500 chars):\")\npreview_prompt = build_variation_prompt(\n    \"Calculus hard-fix\",\n    \"For f(x)=x^3-6x^2+9x+1, find critical points and classify each one.\",\n    \"<think>\\nCompute f'(x)=3x^2-12x+9 and solve f'(x)=0.\\n</think>\\ncritical points: x=1 (maximum), x=3 (minimum).\"\n)\nprint(preview_prompt[:500] + \"...\")

In [4]:
# Extract unique concepts by stripping sequence numbers (#1, #2, etc.)
def normalize_concept(instruction: str) -> str:
    """
    Remove hard-fix sequence numbers from instruction to extract core concept.
    E.g., "Calculus hard-fix #1: ..." -> "Calculus hard-fix"
    """
    return re.sub(r'\s+#\d+:', ':', instruction)

concepts_dict: Dict[str, dict] = defaultdict(list)

for record in raw_data:
    instruction = record["instruction"]
    normalized = normalize_concept(instruction)
    
    # Extract concept label (everything before the colon after hard-fix)
    match = re.match(r'([^:]+):', normalized)
    concept_key = match.group(1).strip() if match else normalized
    
    concepts_dict[concept_key].append({
        "normalized": normalized,
        "original_instruction": instruction,
        "output": record["output"]
    })

print(f"🔍 Extracted {len(concepts_dict)} unique concepts:\n")
for concept, samples in sorted(concepts_dict.items()):
    print(f"  • {concept}: {len(samples)} duplicates")
    print(f"    Template: {samples[0]['normalized']}")

🔍 Extracted 8 unique concepts:

  • Calculus hard-fix: 20 duplicates
    Template: Calculus hard-fix: For f(x)=x^3-6x^2+9x+1, find critical points and classify each one.
  • Code hard-fix: 40 duplicates
    Template: Code hard-fix: Write Python with dfs to detect a cycle in a directed graph and return one cycle path.
  • Instruction hard-fix: 20 duplicates
    Template: Instruction hard-fix: Reply in exactly one sentence about why APIs use request IDs; include trace.
  • Metrics hard-fix: 20 duplicates
    Template: Metrics hard-fix: Given TP=90 FP=30 FN=10, compute precision and recall with decimals.
  • Queue hard-fix: 20 duplicates
    Template: Queue hard-fix: arrivals=120 req/s, service=100 req/s. Estimate backlog growth after 15 minutes.
  • SRE hard-fix: 20 duplicates
    Template: SRE hard-fix: p95=420ms and p99=1200ms. Give exactly two mitigations and expected effect.
  • Safety hard-fix: 20 duplicates
    Template: Safety hard-fix: User asks for phishing help to steal credent

In [3]:
import json
import os
import sys
import re
from pathlib import Path
from typing import Dict, List, Optional
from collections import defaultdict

# Load original dataset
dataset_path = Path("dataset_expert_v3.json")
assert dataset_path.exists(), f"Error: {dataset_path} not found"

with open(dataset_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"✅ Loaded {len(raw_data)} records from {dataset_path}")

# Validate structure
required_fields = {"instruction", "output"}
for i, record in enumerate(raw_data):
    missing = required_fields - set(record.keys())
    if missing:
        print(f"⚠️ Record {i} missing fields: {missing}")
    assert not missing, f"Invalid record at index {i}"

print(f"✅ All records have 'instruction' and 'output' fields")

✅ Loaded 180 records from dataset_expert_v3.json
✅ All records have 'instruction' and 'output' fields


# Dataset Expert V4: Synthetic Data Generation with OpenAI

This notebook transforms `dataset_expert_v3.json` (with massive duplication due to suroverlearning) into `dataset_expert_v4.json` with **genuine diversity** by using OpenAI's API to generate 5 variations per concept.

**Goal:** Avoid overfitting by creating diverse data with different:
- **Numbers**: Different values in math/SRE problems
- **Phrasing**: Different tones (urgent, formal, direct, etc.)
- **Context**: Real-world scenarios and edge cases

**Output:** 50-60 unique training examples instead of 20 repeated ones.